<a href="https://colab.research.google.com/github/aditya01ad/colab_codes/blob/main/notebooks/OR_Prep/OR_prep_M2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Module 2, Topic 1: Integer Programming Foundations**

**Objective:** Formulate problems with discrete decisions using binary and integer variables. Understand when and why integer programming is harder than LP. Build fluency with PuLP's cat='Binary' and cat='Integer'.

### Why integer programming?

LP assumes variables can be fractional. But in reality:

- You can't build 2.5 warehouses

- You can't hire 3.7 employees

- You can't use a route half the time

> Integer Programming (IP) restricts some or all variables to be whole numbers. If all variables are integer -- pure IP. If some are continuous -> Mixed Integer Programming (MIP). Most real OR problems are MIPs.

In [10]:
!pip install "pulp==2.9.0"

In [11]:
from pulp import *
import numpy as np
import pandas as pd

In [12]:
# LP relaxation vs IP: a quick illustration
# Max 3x + 2y sub to x + y <= 4.5, x,y >= 0

# LP version
print("LP:\n")
lp = LpProblem("LP_version", LpMaximize)
x = LpVariable("x", lowBound=0)
y = LpVariable("y", lowBound=0)

lp += 3*x + 2*y
lp +=   x +   y <= 4.5
lp.solve()
print("LP solution: x =", x.varValue, "y =", y.varValue, "obj =", value(lp.objective))

# IP version
print("\nIP:\n")
ip = LpProblem("IP_version", LpMaximize)
xi = LpVariable("x", lowBound=0, cat='Integer')
yi = LpVariable("y", lowBound=0, cat='Integer')

ip += 3*xi + 2*yi
ip +=   xi + 4*yi  <= 4.5
ip.solve()
print("IP solution: x =", xi.varValue, "y =", yi.varValue, "obj =", value(ip.objective))

LP:

LP solution: x = 4.5 y = 0.0 obj = 13.5

IP:

IP solution: x = 4.0 y = 0.0 obj = 12.0


## The knapsack problem — the "hello world" of IP

> Problem: You have a bag with capacity 15 kg. You can choose from 6 items, each with a weight and value. Maximize total value without exceeding capacity.


| Item |	Weight |	Value |
| ---- | ------- | ------ |
| 1	| 2	| 10 |
| 2	| 3	| 15 |
| 3 |	4	| 20 |
| 4	| 5	| 25 |
| 5	| 7	| 35 |
| 6 |	8 |	40 |

### Formulation:

> * Binary variable x_i = 1 if item i is selected
>* Maximize sum(value_i * x_i)
>* Subject to sum(weight_i * x_i) ≤ capacity
>* x_i ∈ {0,1}

In [13]:
# Knapsack
items = ['I1', 'I2', 'I3', 'I4', 'I5', 'I6']
weights = {'I1': 2, 'I2': 3, 'I3': 4, 'I4': 5, 'I5': 6, 'I6': 7}
values = {'I1': 10, 'I2': 15, 'I3': 20, 'I4': 25, 'I5': 35, 'I6': 40}
capacity = 15

knap = LpProblem("Knapsack", LpMaximize)

# Binary decision variables
x = LpVariable.dicts("select", items, cat='Binary')

# Objective
knap+= lpSum(values[i] * x[i] for i in items)

# Capacity constraints
knap += lpSum(weights[i] * x[i] for i in items) <= capacity

knap.solve()

print("Status:", LpStatus[knap.status])
print("Select items:")
total_w, total_v = 0, 0
for i in items:
  if x[i].varValue > 0.5:
    print(f"  {i}: weight={weights[i]}, value={values[i]}")
    total_w += weights[i]
    total_v += values[i]
print(f"Total weight: {total_w}, Total value: {total_v}")

print("knap:", knap)

Status: Optimal
Select items:
  I1: weight=2, value=10
  I5: weight=6, value=35
  I6: weight=7, value=40
Total weight: 15, Total value: 85
knap: Knapsack:
MAXIMIZE
10*select_I1 + 15*select_I2 + 20*select_I3 + 25*select_I4 + 35*select_I5 + 40*select_I6 + 0
SUBJECT TO
_C1: 2 select_I1 + 3 select_I2 + 4 select_I3 + 5 select_I4 + 6 select_I5
 + 7 select_I6 <= 15

VARIABLES
0 <= select_I1 <= 1 Integer
0 <= select_I2 <= 1 Integer
0 <= select_I3 <= 1 Integer
0 <= select_I4 <= 1 Integer
0 <= select_I5 <= 1 Integer
0 <= select_I6 <= 1 Integer



### Step 3: Logical constraints with binary variables –

**Let's go through the standard patterns:**

> Pattern 1: If-then (x=1 → y=1)
If we open a facility (x=1), we must also hire staff (y=1). Model: x ≤ y.

> Pattern 2: Either-or (at most one of two)
We can either build a warehouse in location A or B, but not both. Model: x_A + x_B ≤ 1.

> Pattern 3: At least one
At least one of three suppliers must be selected. Model: x_A + x_B + x_C ≥ 1.

> Pattern 4: Exactly k
Exactly 2 of 5 projects must be chosen. Model: sum(x_i) = 2

**Problem:** A company is selecting 3 projects from 5. Each project has a cost and profit. The budget is ₹50 lakh. At least 2 projects must be selected. Project 1 and Project 3 are mutually exclusive (can't do both). Project 5 requires Project 2 to be selected first.

In [14]:
projects = ['P1', 'P2', 'P3', 'P4', 'P5']
costs = {'P1': 15, 'P2': 20, 'P3':25, 'P4': 10, 'P5': 30}
profits = {'P1': 30, 'P2': 20, 'P3': 45, 'P4': 20, 'P5': 50}
budget = 50

proj = LpProblem("Project_selection", LpMaximize)
x = LpVariable.dicts("select", projects, cat='Binary')

# Objective: maximize profit
proj += lpSum(profits[i] * x[i] for i in projects)

# Budget constraint
proj += lpSum(costs[i] * x[i] for i in projects) <= budget

# At least 2 project
proj += lpSum(x[i] for i in projects) >= 2

# P1 and P3 mutually exclusive
proj += x['P1'] + x['P3'] <= 1

# P5 reuires P2
proj += x['P5']<= x['P2']

proj.solve()

print("Status:", LpStatus[proj.status])
print("Setected projects:")
for i in projects:
  if x[i].varValue > 0.5:
    print(f"  {i}: cost={costs[i]}, profit={profits[i]}")
print("Total Profit =", value(proj.objective))

Status: Optimal
Setected projects:
  P1: cost=15, profit=30
  P2: cost=20, profit=20
  P4: cost=10, profit=20
Total Profit = 70.0


### Step 4: The facility location problem — a classic MIP

**Problem:** We have 3 potential warehouse locations and 5 customers. Each warehouse has a fixed opening cost and a capacity. Each customer has a demand. Shipping cost from warehouse i to customer j is given. Decide which warehouses to open and how to assign customers to minimize total cost (fixed + shipping).

**Formulation:**

>1. Binary y_i = 1 if warehouse i opened
>2. Continuous x_ij = amount shipped from i to j
>3. Minimize sum(fixed_i * y_i) + sum(cost_ij * x_ij)
>4. Subject to:
>    * sum_i x_ij ≥ demand_j for each customer
>    * sum_j x_ij ≤ capacity_i * y_i for each warehouse
>    * x_ij ≥ 0, y_i ∈ {0,1}

In [15]:
np.random.seed(42)
warehouses = ['W1', 'W2', 'W3']
customers = ['C1', 'C2','C3', 'C4', 'C5']

fixed_cost = {'W1': 100, 'W2': 120, 'W3': 90}
capacity = {'W1':150, 'W2': 200, 'W3': 100}
demand = {'C1': 40, 'C2':60, 'C3': 50, 'C4': 70, 'C5': 30}

# Shipping cost matrix
ship_cost = {
        ('W1', 'C1'): 5, ('W1', 'C2'): 7, ('W1', 'C3'): 6, ('W1', 'C4'): 8, ('W1', 'C5'): 4,
        ('W2', 'C1'): 6, ('W2', 'C2'): 5, ('W2', 'C3'): 8, ('W2', 'C4'): 6, ('W2', 'C5'): 7,
        ('W3', 'C1'): 8, ('W3', 'C2'): 6, ('W3', 'C3'): 5, ('W3', 'C4'): 7, ('W3', 'C5'): 6
        }

fac = LpProblem("Facility_Location", LpMinimize)

# Variables
y = LpVariable.dicts("open", warehouses, cat='Binary')
x = LpVariable.dicts("ship", (warehouses, customers), lowBound=0)

# Objective
fac += lpSum(fixed_cost[i] * y[i] for i in warehouses) + \
       lpSum(ship_cost[(i,j)] * x[i][j] for i in warehouses for j in customers)

# Demand satisfaction
for j in customers:
  fac += lpSum(x[i][j] for i in warehouses) >= demand[j], f"Demand_{j}"

# Capacity (only if open)
for i in warehouses:
  fac += lpSum(x[i][j] for j in customers) <= capacity[i] * y[i], f"Capacity_{i}"

fac.solve()

1

In [16]:
print("Status:", LpStatus[fac.status])
print("Open warehouses:")
for i in warehouses:
  if y[i].varValue > 0.5:
    print(f"  {i} (capacity={capacity[i]}, fixed={fixed_cost[i]})")

print("\nShipping plan:")
for i in warehouses:
  for j in customers:
    if x[i][j].varValue > 0.01:
      print(f"  {i} -> {j}: {x[i][j].varValue:.2f} units at cost {ship_cost[(i,j)]}")

print(f"\nTotal cost = Rs.{value(fac.objective):.2f}")

Status: Optimal
Open warehouses:
  W1 (capacity=150, fixed=100)
  W2 (capacity=200, fixed=120)

Shipping plan:
  W1 -> C1: 40.00 units at cost 5
  W1 -> C3: 50.00 units at cost 6
  W1 -> C5: 30.00 units at cost 4
  W2 -> C2: 60.00 units at cost 5
  W2 -> C4: 70.00 units at cost 6

Total cost = Rs.1560.00


> This is a real OR Analyst problem. We'll encounter variations of this in interviews and on the job.



### Step 5: Big-M and linking constraints — deeper look

> The constraint sum_j x_ij <= capacity_i * y_i is a big-M constraint in disguise. When y_i = 0, the RHS is 0, forcing x_ij = 0. When y_i = 1, the RHS is capacity_i.

> **The general form:** x <= M * y, where M is a large enough upper bound on x. Choosing M too large weakens the LP relaxation (slower solve); too small cuts off feasible solutions. Here, capacity_i is the natural tight upper bound.

_The optimal values may be the same, but the loose M takes longer to solve and is a bad habit. Always use the tightest valid M._

In [17]:
# Same facility location but with M = 10000 instead of capacity
fac_loose = LpProblem("Facility_Loose_M", LpMinimize)
y2 = LpVariable.dicts("open", warehouses, cat='Binary')
x2 = LpVariable.dicts("ship", (warehouses, customers), lowBound=0)

fac_loose += lpSum(fixed_cost[i] * y2[i] for i in warehouses) + \
             lpSum(ship_cost[(i,j)] * x2[i][j] for i in warehouses for j in customers)

for j in customers:
  fac_loose += lpSum(x2[i][j] for i in warehouses) >= demand[j]

M= 10000
for i in warehouses:
  fac_loose += lpSum(x2[i][j] for j in customers) <= M * y2[i]

fac_loose.solve()




1

In [18]:
print("Loose M total ocst:", value(fac_loose.objective))
print("Tight M total cost:", value(fac.objective))

Loose M total ocst: 1560.0
Tight M total cost: 1560.0


### Step 6: Branch-and-bound intuition (conceptual, not coded)

PuLP uses a solver (CBC by default) that runs branch-and-bound. Here's what happens under the hood:

1. Solve LP relaxation (ignore integrality). Get a bound.

2. If solution is integer, done.

3. If fractional, pick a fractional variable (say y_1 = 0.6). Branch into two subproblems:

    - Subproblem A: y_1 = 0

    - Subproblem B: y_1 = 1

4. Solve each subproblem's LP relaxation. If the bound is worse than the current best integer solution, prune (fathom) it.

5. Continue branching until all subproblems are pruned or solved to integrality.

# Step 6: Branch-and-Bound Intuition — Full Walkthrough

A conceptual, worked example of how integer programming solvers actually find optimal solutions. Read once, then read again with a pencil — draw the tree yourself. This is the kind of understanding that gets you through interviews at Amazon, Uber, Delhivery, and any serious OR team.

---

## The problem: a small integer program

We'll use a small knapsack-style IP so the numbers stay manageable.

**Maximize** 8x₁ + 11x₂ + 6x₃ + 4x₄  
**Subject to:**  
5x₁ + 7x₂ + 4x₃ + 3x₄ ≤ 14  
x₁, x₂, x₃, x₄ ∈ {0, 1}

This is a 4-variable binary knapsack. Brute force would need 16 combinations — trivial. But the point is to see **how branch-and-bound reasons** about it, because the same logic scales to thousands of variables.

---

## Step 1: Solve the LP relaxation

Relax the integrality constraint. Allow xᵢ to be continuous between 0 and 1.

**LP relaxation:**  
Maximize 8x₁ + 11x₂ + 6x₃ + 4x₄  
Subject to 5x₁ + 7x₂ + 4x₃ + 3x₄ ≤ 14  
0 ≤ x₁, x₂, x₃, x₄ ≤ 1

Solve by **profit density** (profit per unit of constraint):

| Variable | Profit | Weight | Density |
|----------|--------|--------|---------|
| x₁       | 8      | 5      | 1.60    |
| x₂       | 11     | 7      | 1.57    |
| x₃       | 6      | 4      | 1.50    |
| x₄       | 4      | 3      | 1.33    |

Greedy fill by density:
- x₁ = 1 (uses 5, remaining 9)
- x₂ = 1 (uses 7, remaining 2)
- x₃ = 2/4 = 0.5 (uses remaining 2)
- x₄ = 0

**LP optimum:** (1, 1, 0.5, 0), objective = 8 + 11 + 3 = **22**.

But x₃ = 0.5 is fractional, so this isn't a valid IP solution. **Branch.**

---

## Step 2: The branch-and-bound tree

### Node 0 (root): LP relaxation
- Solution: x₁=1, x₂=1, x₃=0.5, x₄=0
- Objective: **22** (upper bound for any IP solution)
- Status: fractional → branch on x₃

---

### Node 1: Branch x₃ = 0
Add constraint x₃ = 0.

**Maximize** 8x₁ + 11x₂ + 4x₄  
**Subject to:** 5x₁ + 7x₂ + 3x₄ ≤ 14, x₁, x₂, x₄ ∈ [0,1]

Greedy: x₁ = 1 (5), x₂ = 1 (7, total 12), x₄ = 2/3 (remaining 2 of 3).
Objective = 8 + 11 + 4(2/3) ≈ 21.67.

- Solution: (1, 1, 0, 0.667)
- **Upper bound = 21.67**
- Fractional → branch on x₄

---

### Node 2: Branch x₃ = 1
Add constraint x₃ = 1. Consumes 4 units, remaining capacity = 10.

**Maximize** 8x₁ + 11x₂ + 6 + 4x₄  
**Subject to:** 5x₁ + 7x₂ + 3x₄ ≤ 10

Greedy: x₁ = 1 (5), x₂ = 5/7 ≈ 0.714 (uses remaining 5 of 7).
Objective = 8 + 11(5/7) + 6 ≈ 21.86.

- Solution: (1, 0.714, 1, 0)
- **Upper bound = 21.86**
- Fractional → branch on x₂

---

### Sub-branches from Node 1 (x₃ = 0)

**Node 1a: x₃ = 0, x₄ = 0**  
Remaining: 5x₁ + 7x₂ ≤ 14.  
LP: x₁ = 1 (5), x₂ = 1 (7, total 12). **Integer!**  
Objective = 8 + 11 = **19**.
- **Feasible integer solution: (1, 1, 0, 0), obj = 19**
- This becomes our **incumbent** (best known integer solution).
- No need to branch. Fathomed.

**Node 1b: x₃ = 0, x₄ = 1**  
Remaining: 5x₁ + 7x₂ ≤ 11.  
LP: x₁ = 1 (5), x₂ = 6/7 ≈ 0.857.  
Objective = 8 + 11(6/7) ≈ 17.43. Fractional.
- Upper bound = 17.43.
- Since 17.43 < 19 (our incumbent), **prune this branch**. Even the best possible in this subtree can't beat 19.

---

### Sub-branches from Node 2 (x₃ = 1)

**Node 2a: x₃ = 1, x₂ = 0**  
Remaining: 5x₁ + 3x₄ ≤ 10.  
LP: x₁ = 1 (5), x₄ = 1 (3, total 8, still feasible). **Integer!**  
Objective = 8 + 0 + 6 + 4 = **18**.
- Integer solution: (1, 0, 1, 1), obj = **18**.
- 18 < 19 (incumbent), so no improvement. Fathomed.

**Node 2b: x₃ = 1, x₂ = 1**  
Consumes 5 + 7 + 4 = 16 > 14. **Infeasible.** Prune.

---

## Step 3: The final tree (visual)

```
                                  [Node 0]
                           LP: 22, (1, 1, 0.5, 0)
                                 /        \
                             x₃=0          x₃=1
                             /                \
                     [Node 1]                  [Node 2]
                    UB: 21.67                 UB: 21.86
                 (1, 1, 0, 0.667)          (1, 0.714, 1, 0)
                   /       \                /        \
                x₄=0       x₄=1          x₂=0        x₂=1
                 /           \             |            \
            [Node 1a]      [Node 1b]    [Node 2a]    [Node 2b]
             OBJ: 19        UB: 17.43     OBJ: 18      INFEASIBLE
             INT ✓          PRUNED        INT ✓        PRUNED
           incumbent
```

**Optimal IP solution:** (x₁, x₂, x₃, x₄) = (1, 1, 0, 0), objective = **19**.

---

## Step 4: The logic in plain English

1. **Start with LP relaxation.** Its objective (22) is an **upper bound** on the true IP optimum, because relaxing constraints can only help. The IP answer cannot exceed 22.
2. **If LP solution is integer**, you're done. It's optimal for the IP too.
3. **If fractional**, pick one fractional variable and branch: two child problems, one forcing it to 0, the other to 1. Every IP solution must fall into one of these two cases, so no solution is lost.
4. **For each child, solve its LP relaxation** to get an upper bound for that subtree.
5. **Prune a subtree** if any of these hold:
   - **Infeasible:** no solution exists (like Node 2b).
   - **Integer:** the LP solution is integer — it's the best in that subtree, record it as the new incumbent if better.
   - **Bounded:** the subtree's LP bound is worse than the best known integer (incumbent). No point exploring (like Node 1b).
6. **Keep branching** until every subtree is pruned. The incumbent at the end is the optimal IP solution.

---

## Step 5: Why this matters for OR Analyst interviews

- **Solver speed varies wildly.** A poorly formulated model has loose LP bounds → many nodes → slow solve. A tight formulation prunes aggressively → fast solve. This is why "tight big-M" matters.
- **NP-hardness in practice.** The tree can explode exponentially. For 50 binaries, the tree might have billions of nodes in the worst case. Heuristics exist because of this.
- **Every MIP you've solved in PuLP uses this.** CBC (the default solver) runs branch-and-bound behind the scenes. Now you know what's happening.
- **Likely interview question:** "What happens when you solve an IP? Why does it take longer than LP?"
  - **Your answer:** "The solver relaxes integrality to get a bound, then branches on fractional variables, pruning subtrees whose bounds are worse than the best integer found so far. Formulation quality determines how tight the bounds are, and hence how fast it solves."

---

## Step 6 (Optional): Watch the tree in action

If you want to see CBC's branch-and-bound log, run this:

```python
from pulp import *

prob = LpProblem("B&B_Demo", LpMaximize)
x1 = LpVariable("x1", cat='Binary')
x2 = LpVariable("x2", cat='Binary')
x3 = LpVariable("x3", cat='Binary')
x4 = LpVariable("x4", cat='Binary')

prob += 8*x1 + 11*x2 + 6*x3 + 4*x4
prob += 5*x1 + 7*x2 + 4*x3 + 3*x4 <= 14

prob.solve(PULP_CBC_CMD(msg=1))  # msg=1 shows solver log

print("Status:", LpStatus[prob.status])
print(f"x1={x1.varValue}, x2={x2.varValue}, x3={x3.varValue}, x4={x4.varValue}")
print("Objective:", value(prob.objective))
```

You'll see lines like `Node 1: ... Best Possible ...` in the log. That's the branch-and-bound tree being explored in real time. Match those lines to the tree diagram above — it's satisfying when it clicks.

---

## Key takeaway

> Branch-and-bound works by relaxing integrality to get optimistic bounds, then systematically eliminating regions of the search space that provably cannot contain a better solution than the best integer found so far. The tighter the LP bound, the faster the search. This is why formulation quality is a competitive skill in OR — the same problem can solve in milliseconds or hours depending on how you write the constraints.

### When does IP become hard?

- Knapsack: NP-hard, but small instances solve quickly.

- TSP: NP-hard, 20 cities is easy, 100 cities is hard, 1000 cities requires heuristics.

- Facility location: NP-hard, but modern solvers handle 100s of facilities with good formulations.

- VRP: NP-hard, real-world instances (1000+ customers) require heuristics or column generation.

> For your interviews, know: IP is NP-hard in general; solvers use branch-and-bound with LP relaxation bounds; formulation quality (tight M, strong constraints) dramatically affects solve time.

# Module 2, Topic 1: Integer Programming Foundations — Session Notes

**Date:** Aug 6, 2026  
**Status:** Completed ✅

## Concepts covered
- LP vs IP: integrality restricts feasible region, objective worsens
- Binary variables: `cat='Binary'` in PuLP
- Integer variables: `cat='Integer'`
- Knapsack: classic binary IP
- Logical constraints: if-then (x ≤ y), either-or (x+y ≤ 1), at-least-one (sum ≥ 1), exactly-k (sum = k)
- Facility location: fixed costs + assignment, capacity linking constraints
- Big-M: tight bounds matter for solver performance
- Branch-and-bound: LP relaxation → branching → pruning

## Key insights
- Binary variables model yes/no decisions; integer variables model counts.
- Logical conditions translate to linear constraints with binaries.
- Formulation quality (tight M, valid inequalities) affects solve time.
- Most real OR problems are MIPs, not pure LPs.

## Next steps
- Practice more formulations: scheduling, bin packing, blending with discrete choices.
- Move to heuristics for large-scale IPs.